# NumCompute - End-to-End Quickstart Demo

This notebook demonstrates the full NumCompute toolkit built from scratch using **plain Python + NumPy only**.

### Modules covered
| Module | Purpose |
|---|---|
| `io.py` | CSV reading with missing value handling |
| `preprocessing.py` | Scaling, imputation, encoding |
| `sort_search.py` | Sorting, top-k, binary search |
| `rank.py` | Ranking with tie handling, percentiles |
| `stats.py` | Descriptive statistics, histogram, quantiles |
| `optim.py` | Finite-difference gradients and Jacobian |
| `pipeline.py` | Chaining transformers into a pipeline |
| `benchmarking.py` | Vectorised vs loop performance comparison |
| `metrics.py` | Accuracy, Precision, Recall, F1, MSE, ROC/AUC |
| `utils.py` | BaseEstimator, BaseTransformer, distances, activations |

> **Dataset used:** `Iris.csv`, `airports.csv`, `sales_data.csv`

---
## 0. Setup - Imports

In [1]:
import sys
import os
import numpy as np

# Make sure the root folder is on the path so numcompute can be imported
sys.path.insert(0, os.path.abspath('..'))

print('NumPy version:', np.__version__)
print('Setup complete.')

NumPy version: 2.0.2
Setup complete.


---
## 1. Data I/O - Reading CSV Files

Using `io.py` to load CSV files into NumPy arrays.  
The loader handles **missing values**, custom delimiters, and dtype conversion.

In [2]:
from numcompute.io import load_csv

# Load Iris dataset
iris_data = load_csv('../data/Iris.csv')
print('Iris dataset loaded.')
print('Shape:', iris_data.shape)
print('First 5 rows:\n', iris_data[:5])

Iris dataset loaded.
Shape: (150, 6)
First 5 rows:
 [[1.  5.1 3.5 1.4 0.2 nan]
 [2.  4.9 3.  1.4 0.2 nan]
 [3.  4.7 3.2 1.3 0.2 nan]
 [4.  4.6 3.1 1.5 0.2 nan]
 [5.  5.  3.6 1.4 0.2 nan]]


---
## 2. Preprocessing

Using `preprocessing.py` to:
- **Impute** missing values (replace NaN with column mean)
- **StandardScaler** - zero mean, unit variance
- **MinMaxScaler** - scale to [0, 1] range
- **OneHotEncoder** - encode categorical columns


In [3]:
from numcompute.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

# Use numeric columns from Iris (columns 0-3: sepal/petal length & width)
X_iris = iris_data[:, :4].astype(float)
print('Raw Iris features (first 5 rows):\n', X_iris[:5])

Raw Iris features (first 5 rows):
 [[1.  5.1 3.5 1.4]
 [2.  4.9 3.  1.4]
 [3.  4.7 3.2 1.3]
 [4.  4.6 3.1 1.5]
 [5.  5.  3.6 1.4]]


In [4]:
# StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)
print('After StandardScaler (first 5 rows):\n', X_scaled[:5])
print('Column means (should be ~0):', np.round(X_scaled.mean(axis=0), 4))
print('Column stds  (should be ~1):', np.round(X_scaled.std(axis=0), 4))

After StandardScaler (first 5 rows):
 [[-1.72054204 -0.90068117  1.03205722 -1.3412724 ]
 [-1.69744751 -1.14301691 -0.1249576  -1.3412724 ]
 [-1.67435299 -1.38535265  0.33784833 -1.39813811]
 [-1.65125846 -1.50652052  0.10644536 -1.2844067 ]
 [-1.62816394 -1.02184904  1.26346019 -1.3412724 ]]
Column means (should be ~0): [ 0. -0. -0. -0.]
Column stds  (should be ~1): [1. 1. 1. 1.]


In [5]:
# MinMaxScaler
minmax = MinMaxScaler()

X_minmax = minmax.fit_transform(X_iris)
print('After MinMaxScaler (first 5 rows):\n', X_minmax[:5])
print('Column min (should be 0):', X_minmax.min(axis=0))
print('Column max (should be 1):', X_minmax.max(axis=0))

After MinMaxScaler (first 5 rows):
 [[0.         0.22222222 0.625      0.06779661]
 [0.00671141 0.16666667 0.41666667 0.06779661]
 [0.01342282 0.11111111 0.5        0.05084746]
 [0.02013423 0.08333333 0.45833333 0.08474576]
 [0.02684564 0.19444444 0.66666667 0.06779661]]
Column min (should be 0): [0. 0. 0. 0.]
Column max (should be 1): [1. 1. 1. 1.]


In [6]:
# OneHotEncoder

encoder = OneHotEncoder()

X_categories = np.array([
            ['cat', 1],
            ['dog', 2],
            ['cat', 3]
        ], dtype=object) 

X_encoded = encoder.fit_transform(X_categories)
print("Expect binary values for each feature:\n", X_encoded)

Expect binary values for each feature:
 [[1. 0. 1. 0. 0.]
 [0. 1. 0. 1. 0.]
 [1. 0. 0. 0. 1.]]


---
## 3. Sorting, Search & Top-K

Using `sort_search.py` to demonstrate:
- **Stable sort** on array columns
- **Top-K** largest values using `np.argpartition`
- **Binary search** for fast element lookup


In [7]:
from numcompute.sort_search import topk, binary_search, stable_sort

# Use the first column of scaled Iris data
col = X_scaled[:, 0]
print('First column (first 10 values):', col[:10])

First column (first 10 values): [-1.72054204 -1.69744751 -1.67435299 -1.65125846 -1.62816394 -1.60506942
 -1.58197489 -1.55888037 -1.53578584 -1.51269132]


In [ ]:
# Create data with duplicate values to demonstrate stability
# Format: [Value, Original Index]
data_with_ties = np.array([
    [2, 0],
    [1, 1],
    [2, 2],
    [1, 3]
])

# Sort by the first column (index 0)
# A stable sort ensures that the row with index 1 stays before the row with index 3
sorted_stable = np.sort(data_with_ties, axis=0, kind='stable')

print("Original Data (Value, Original Pos):\n", data_with_ties)
print("\nStable Sorted Data:\n", sorted_stable)

Original Data (Value, Original Pos):
 [[2 0]
 [1 1]
 [2 2]
 [1 3]]

Stable Sorted Data:
 [[1 0]
 [1 1]
 [2 2]
 [2 3]]


In [10]:
# Top-K largest values
k = 5
top_values, top_indices = topk(col, k=k, largest=True, return_indices=True)
print(f'Top {k} largest values:', top_values)
print(f'Their indices:', top_indices)

Top 5 largest values: [1.72054204 1.69744751 1.67435299 1.65125846 1.62816394]
Their indices: [149 148 147 146 145]


In [11]:
# Binary search
sorted_col = np.sort(col)
target = sorted_col[10]   # pick a known value
idx, found = binary_search(sorted_col, target)
print(f'Searching for {target:.4f} in sorted array')
print(f'Found: {found}  |  Index: {idx}')

Searching for -1.4896 in sorted array
Found: True  |  Index: 10


---
## 4. Ranking

Using `rank.py` to demonstrate:
- **Ranking** with tie-handling methods (average, dense, ordinal)
- **Percentiles** of a data column

In [29]:
from numcompute.rank import rank, percentile

# Small example to clearly show tie handling
sample = np.array([3.0, 1.0, 4.0, 1.0, 5.0, 1.0, 2.0])
print('Sample data:', sample)

Sample data: [3. 1. 4. 1. 5. 1. 2.]


In [30]:
# Ranking with tie methods
for method in ['average', 'dense', 'ordinal']:
    ranks = rank(sample, method=method)
    print(f'Rank ({method:>8}): {ranks}')

Rank ( average): [5. 2. 6. 2. 7. 2. 4.]
Rank (   dense): [3 1 4 1 5 1 2]
Rank ( ordinal): [5. 1. 6. 2. 7. 3. 4.]


In [31]:
# Percentiles
col = X_scaled[:, 0]
for q in [25, 50, 75]:
    p = percentile(col, q=q)
    print(f'P{q}: {p:.4f}')

P25: -0.8603
P50: 0.0000
P75: 0.8603


---
## 5. Descriptive Statistics

Using `stats.py` to compute:
- Mean, median, std, min, max
- Quantiles (Q25, Q50, Q75)
- Histogram bin counts

In [32]:
from numcompute.stats import mean, median, std, minimum, maximum, quantiles, histogram, stats

col = X_scaled[:, 0]
print('=== Descriptive Stats - Iris Feature 0 (StandardScaled) ===')
print(f'Mean    : {mean(col):.4f}')
print(f'Median  : {median(col):.4f}')
print(f'Std Dev : {std(col):.4f}')
print(f'Min     : {minimum(col):.4f}')
print(f'Max     : {maximum(col):.4f}')

=== Descriptive Stats - Iris Feature 0 (StandardScaled) ===
Mean    : 0.0000
Median  : 0.0000
Std Dev : 1.0000
Min     : -1.7205
Max     : 1.7205


In [33]:
# Quantiles
q25, q50, q75 = quantiles(col, q=(0.25, 0.50, 0.75))
print(f'Q25: {q25:.4f}  |  Q50: {q50:.4f}  |  Q75: {q75:.4f}')

Q25: -0.8603  |  Q50: 0.0000  |  Q75: 0.8603


In [34]:
# Histogram
counts, edges = histogram(col, bins=8)
print('Histogram bin counts:', counts)
print('Bin edges:', np.round(edges, 3))

Histogram bin counts: [19 19 18 19 19 18 19 19]
Bin edges: [-1.721 -1.29  -0.86  -0.43   0.     0.43   0.86   1.29   1.721]


In [35]:
# Full stats summary dict
summary = stats(col)
print('\nFull stats summary:')
for key, val in summary.items():
    print(f'  {key:>8}: {val}')


Full stats summary:
      mean: 0.0
    median: 0.0
       std: 1.0
       min: -1.7205420368774056
       max: 1.7205420368774056
       q25: -0.8602710184387028
       q75: 0.8602710184387028


---
## 6. Finite-Difference Gradients (optim.py)

Using `optim.py` to estimate gradients numerically using finite differences.

- **grad** - gradient of a scalar function
- **jacobian** - Jacobian matrix of a vector function

In [36]:
from numcompute.optim import grad, jacobian

# Scalar function: f(x) = x0² + 2·x1² + 3·x2
def scalar_fn(x):
    return x[0]**2 + 2*x[1]**2 + 3*x[2]

x0 = np.array([1.0, 2.0, 3.0])

g = grad(scalar_fn, x0)
print('Function : f(x) = x0² + 2x1² + 3x2')
print('At x     :', x0)
print('Gradient (finite diff)  :', np.round(g, 6))
print('Gradient (analytical)   :', np.array([2*x0[0], 4*x0[1], 3.0]))

Function : f(x) = x0² + 2x1² + 3x2
At x     : [1. 2. 3.]
Gradient (finite diff)  : [2. 8. 3.]
Gradient (analytical)   : [2. 8. 3.]


In [37]:
# Vector function: Jacobian
def vector_fn(x):
    return np.array([x[0]**2, x[0]*x[1], x[1]**2])

x0 = np.array([2.0, 3.0])

J = jacobian(vector_fn, x0)
print('Function : [x0², x0·x1, x1²]')
print('At x     :', x0)
print('Jacobian (finite diff):\n', np.round(J, 6))
print('Jacobian (analytical):\n', np.array([[2*x0[0], 0], [x0[1], x0[0]], [0, 2*x0[1]]]))

Function : [x0², x0·x1, x1²]
At x     : [2. 3.]
Jacobian (finite diff):
 [[4. 0.]
 [3. 2.]
 [0. 6.]]
Jacobian (analytical):
 [[4. 0.]
 [3. 2.]
 [0. 6.]]


---
## 7. Pipeline - Chaining Transformers

Using `pipeline.py` to chain **Imputer → StandardScaler** into a single reusable pipeline.

This means instead of calling each step manually, we just call `pipe.fit_transform(X)` once.

In [38]:
from numcompute.pipeline import Pipeline, FeatureUnion
from numcompute.preprocessing import StandardScaler, Imputer

# Build a pipeline: Impute missing values → then StandardScale
pipe = Pipeline([
    ('impute', Imputer()),
    ('scale',  StandardScaler()),
])

print('Pipeline created:')
print(pipe)

Pipeline created:
Pipeline(steps=[
  ('impute', <numcompute.preprocessing.Imputer object at 0x0000017F0C83BBE0>)
  ('scale', <numcompute.preprocessing.StandardScaler object at 0x0000017F0C83BC40>)
])


In [39]:
# Fit and transform training data
X_iris = iris_data[:, :4].astype(float)

# Introduce some fake NaNs to show imputation works
X_with_nan = X_iris.copy()
X_with_nan[0, 0] = np.nan
X_with_nan[5, 2] = np.nan
X_with_nan[10, 1] = np.nan

print('Input has NaNs:', np.isnan(X_with_nan).any())

X_out = pipe.fit_transform(X_with_nan)

print('After pipeline - NaNs remaining:', np.isnan(X_out).any())
print('Output shape:', X_out.shape)
print('First 5 rows:\n', np.round(X_out[:5], 4))

Input has NaNs: True
After pipeline - NaNs remaining: False
Output shape: (150, 4)
First 5 rows:
 [[-1.743  -0.7431  0.9566 -1.3413]
 [-1.6968 -0.9532 -0.0567 -1.3413]
 [-1.6738 -1.1633  0.3486 -1.3981]
 [-1.6507 -1.2683  0.1459 -1.2844]
 [-1.6276 -0.8481  1.1592 -1.3413]]


In [40]:
# Apply same learned pipeline to new (test) data
X_test = X_iris[140:].copy()   # last 10 rows as test
X_test_out = pipe.transform(X_test)
print('Test data transformed (last 10 rows):\n', np.round(X_test_out, 4))

Test data transformed (last 10 rows):
 [[ 1.5124  0.9378  0.1459  1.0471]
 [ 1.5355  1.1479  0.1459  0.7628]
 [ 1.5586 -0.0077 -0.6647  0.7628]
 [ 1.5817  1.0428  0.3486  1.2177]
 [ 1.6048  0.9378  0.5512  1.104 ]
 [ 1.6279  0.9378 -0.0567  0.8196]
 [ 1.651   0.5176 -1.07    0.7059]
 [ 1.6741  0.7277 -0.0567  0.8196]
 [ 1.6971  0.4125  0.7539  0.9334]
 [ 1.7202  0.0973 -0.0567  0.7628]]


In [41]:
# FeatureUnion: run two scalers in parallel, combine outputs
from numcompute.preprocessing import MinMaxScaler

union = FeatureUnion([
    ('standard', StandardScaler()),
    ('minmax',   MinMaxScaler()),
])

X_union = union.fit_transform(X_iris)
print('FeatureUnion output shape:', X_union.shape)  # should be (150, 8)
print('First 3 rows:\n', np.round(X_union[:3], 4))

FeatureUnion output shape: (150, 8)
First 3 rows:
 [[-1.7205 -0.9007  1.0321 -1.3413  0.      0.2222  0.625   0.0678]
 [-1.6974 -1.143  -0.125  -1.3413  0.0067  0.1667  0.4167  0.0678]
 [-1.6744 -1.3854  0.3378 -1.3981  0.0134  0.1111  0.5     0.0508]]


---
## 8. Benchmarking - Vectorised vs Python Loops

Using `benchmarking.py` to compare the speed of **NumPy vectorised operations** vs equivalent **Python loop** implementations.

This demonstrates why vectorisation is a core design goal of NumCompute.

In [42]:
from numcompute.benchmarking import compare, BenchmarkSuite, run_default_benchmarks

# Quick single comparison
X_bench = np.random.default_rng(42).standard_normal((3000, 20))

def loop_mean():
    result = np.empty(X_bench.shape[1])
    for j in range(X_bench.shape[1]):
        total = 0.0
        for i in range(X_bench.shape[0]):
            total += X_bench[i, j]
        result[j] = total / X_bench.shape[0]
    return result

result = compare(
    label='Column mean (3000×20)',
    vectorised_fn=lambda: np.mean(X_bench, axis=0),
    loop_fn=loop_mean,
    n_runs=50,
)


  Benchmark: Column mean (3000×20)
-----------------------------------------------------------------
  Metric                    Vectorised            Loop
-----------------------------------------------------------------
  Mean time                  0.1025 ms      15.3709 ms
  Std dev                    0.0151 ms       5.9369 ms
  Min time                   0.0850 ms       9.3026 ms
  Max time                   0.1918 ms      33.4394 ms
  Runs                              50              50
-----------------------------------------------------------------
  Speedup: Vectorised is 149.90x faster



In [43]:
# Full benchmark suite
suite = run_default_benchmarks(n_rows=3000, n_cols=20, n_runs=50)


#################################################################
  NumCompute Default Benchmark Suite
  Array shape: (3000, 20)  |  n_runs: 50
#################################################################

  Benchmark: Column mean
-----------------------------------------------------------------
  Metric                    Vectorised            Loop
-----------------------------------------------------------------
  Mean time                  0.0948 ms      11.5389 ms
  Std dev                    0.0129 ms       2.4759 ms
  Min time                   0.0603 ms       8.7893 ms
  Max time                   0.1303 ms      19.9994 ms
  Runs                              50              50
-----------------------------------------------------------------
  Speedup: Vectorised is 121.76x faster


  Benchmark: Column std dev
-----------------------------------------------------------------
  Metric                    Vectorised            Loop
--------------------------------------------

### Performance Summary

The table above shows the speedup of vectorised NumPy operations over equivalent Python loops.

Key observations:
- **Softmax and dot product** show the largest speedup because NumPy delegates to optimised BLAS routines
- **Sorting** benefits from NumPy's C-level introsort vs Python's interpreted loop
- Even simple operations like **column mean** show significant speedup at scale

This confirms that NumCompute's vectorisation-first design is well justified.

---
## 9. Evaluation Metrics (metrics.py)

Using `metrics.py` to evaluate model performance with:
- **Classification:** Accuracy, Precision, Recall, F1, Confusion Matrix
- **Regression:** Mean Squared Error (MSE)
- **Bonus:** ROC Curve and AUC score


In [44]:
from numcompute.metrics import (
    accuracy, precision, recall, f1,
    confusion_matrix, mse, roc_curve, auc
)

# Synthetic binary classification results
# 1 = positive class, 0 = negative class
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 0])

print('True labels     :', y_true)
print('Predicted labels:', y_pred)

True labels     : [1 0 1 1 0 1 0 0 1 0]
Predicted labels: [1 0 1 0 0 1 1 0 1 0]


In [54]:
# Classification metrics
print('=== Classification Metrics ===')
print(f'Accuracy  : {accuracy(y_true, y_pred):.4f}')
print(f'Precision : {precision(y_true, y_pred):.4f}')
print(f'Recall    : {recall(y_true, y_pred):.4f}')
print(f'F1 Score  : {f1(y_true, y_pred):.4f}')

=== Classification Metrics ===
Accuracy  : 0.8000
Precision : 0.8000
Recall    : 0.8000
F1 Score  : 0.8000


In [46]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print('Confusion Matrix:')
print('               Predicted 0   Predicted 1')
print(f'  Actual 0  :     {cm[0,0]}             {cm[0,1]}')
print(f'  Actual 1  :     {cm[1,0]}             {cm[1,1]}')

Confusion Matrix:
               Predicted 0   Predicted 1
  Actual 0  :     4             1
  Actual 1  :     1             4


In [47]:
# Regression metric: MSE
y_true_reg = np.array([3.0, 2.5, 4.0, 5.0, 1.5])
y_pred_reg = np.array([2.8, 2.7, 3.8, 5.2, 1.4])

print('=== Regression Metrics ===')
print(f'MSE : {mse(y_true_reg, y_pred_reg):.6f}')

=== Regression Metrics ===
MSE : 0.034000


In [48]:
# ROC Curve and AUC
# y_scores = predicted probabilities (not hard labels)
y_scores = np.array([0.9, 0.2, 0.85, 0.4, 0.1, 0.95, 0.6, 0.15, 0.8, 0.3])

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
auc_score = auc(fpr, tpr)

print('=== ROC / AUC ===')
print(f'AUC Score : {auc_score:.4f}  (1.0 = perfect, 0.5 = random)')
print(f'FPR values: {np.round(fpr, 3)}')
print(f'TPR values: {np.round(tpr, 3)}')

=== ROC / AUC ===
AUC Score : 0.9600  (1.0 = perfect, 0.5 = random)
FPR values: [0.  0.  0.  0.  0.  0.2 0.2 0.4 0.6 0.8 1. ]
TPR values: [0.  0.2 0.4 0.6 0.8 0.8 1.  1.  1.  1.  1. ]


---
## End-to-End Summary

This notebook demonstrated the complete NumCompute toolkit:

1. **io.py** - loaded real CSV datasets including handling of missing values
2. **preprocessing.py** - imputed NaNs, applied StandardScaler and MinMaxScaler
3. **sort_search.py** - found top-K values and performed binary search
4. **rank.py** - ranked data with multiple tie-handling strategies and computed percentiles
5. **stats.py** - computed full descriptive statistics including histograms and quantiles
6. **optim.py** - estimated gradients and Jacobians via finite differences
7. **pipeline.py** - chained transformers into reusable pipelines and FeatureUnions
8. **benchmarking.py** - proved vectorised implementations are significantly faster than Python loops
9. **metrics.py** - evaluated classification (Accuracy, F1, ROC/AUC) and regression (MSE)